# Returns Incremental Processing

## File Parameter

In [ ]:
file_name = ""

## Define Returns Schema

In [5]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DateType,
    DecimalType,
    StringType,
    TimestampType
)

returns_schema = StructType([
    StructField("ReturnID", IntegerType(), True),
    StructField("SaleID", IntegerType(), True),
    StructField("ReturnDate", StringType(), True),
    StructField("ReturnQuantity", IntegerType(), True),
    StructField("ReturnAmount", DecimalType(18, 2), True),
    StructField("ReturnReason", StringType(), True),
    StructField("ModifiedDate", StringType(), True),
    StructField("CreatedDate", StringType(), True),
    StructField("UpdatedDate", StringType(), True)
])

StatementMeta(, 189e39be-bc0a-4371-b963-425c8927f739, 7, Finished, Available, Finished, False)

## Read Incremental CSV File

In [ ]:
df_returns = spark.read \
    .option("header", "true") \
    .schema(returns_schema) \
    .csv(f"Files/Returns/{file_name}")



In [ ]:
from pyspark.sql.functions import col, to_date, to_timestamp, coalesce

df_returns = df_returns \
    .withColumn(
        "ReturnDate",
        coalesce(
            to_date(col("ReturnDate"), "dd-MM-yyyy"),
            to_date(col("ReturnDate"), "yyyy-MM-dd")
        )
    ) \
    .withColumn(
        "ModifiedDate",
        coalesce(
            to_timestamp(col("ModifiedDate"), "dd-MM-yyyy HH:mm"),
            to_timestamp(col("ModifiedDate"), "yyyy-MM-dd HH:mm:ss")
        )
    ) \
    .withColumn(
        "CreatedDate",
        coalesce(
            to_timestamp(col("CreatedDate"), "dd-MM-yyyy HH:mm"),
            to_timestamp(col("CreatedDate"), "yyyy-MM-dd HH:mm:ss")
        )
    ) \
    .withColumn(
        "UpdatedDate",
        coalesce(
            to_timestamp(col("UpdatedDate"), "dd-MM-yyyy HH:mm"),
            to_timestamp(col("UpdatedDate"), "yyyy-MM-dd HH:mm:ss")
        )
    )

## Create Temporary SQL View

In [8]:
df_returns.createOrReplaceTempView("returns_source")

StatementMeta(, 189e39be-bc0a-4371-b963-425c8927f739, 10, Finished, Available, Finished, False)

## Incremental Upsert into FactReturns

In [20]:
spark.sql("""
MERGE INTO factreturns AS target
USING returns_source AS source
ON target.ReturnID = source.ReturnID

WHEN MATCHED AND source.UpdatedDate > target.UpdatedDate THEN
  UPDATE SET
    target.SaleID = source.SaleID,
    target.ReturnDate = source.ReturnDate,
    target.ReturnQuantity = source.ReturnQuantity,
    target.ReturnAmount = source.ReturnAmount,
    target.ReturnReason = source.ReturnReason,
    target.ModifiedDate = source.ModifiedDate,
    target.CreatedDate = source.CreatedDate,
    target.UpdatedDate = source.UpdatedDate

WHEN NOT MATCHED THEN
  INSERT (
    ReturnID,
    SaleID,
    ReturnDate,
    ReturnQuantity,
    ReturnAmount,
    ReturnReason,
    ModifiedDate,
    CreatedDate,
    UpdatedDate
  )
  VALUES (
    source.ReturnID,
    source.SaleID,
    source.ReturnDate,
    source.ReturnQuantity,
    source.ReturnAmount,
    source.ReturnReason,
    source.ModifiedDate,
    source.CreatedDate,
    source.UpdatedDate
  )
""")

StatementMeta(, 189e39be-bc0a-4371-b963-425c8927f739, 22, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]